In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/digit-recognizer/sample_submission.csv
/kaggle/input/digit-recognizer/train.csv
/kaggle/input/digit-recognizer/test.csv


## Loading the Dataset

In [2]:
train_df = pd.read_csv('/kaggle/input/digit-recognizer/train.csv')
test_df = pd.read_csv('/kaggle/input/digit-recognizer/test.csv')

In [3]:
train_df.head()

,label,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [4]:
train_df['label'].value_counts()

1    4684
7    4401
3    4351
9    4188
2    4177
6    4137
0    4132
4    4072
8    4063
5    3795
Name: label, dtype: int64

In [5]:
X = train_df.drop(['label'],axis=1)
y = train_df['label']

In [6]:
X = X/255

In [7]:
# X['pixel50'].max()

## Splitting the Dataset

In [8]:
from sklearn.model_selection import train_test_split

X_train,X_val,y_train,y_val = train_test_split(X,y,test_size=0.2,random_state=42)

In [9]:
X_train.shape,X_val.shape

((33600, 784), (8400, 784))

## Training ML Models

### 1. K-Nearest Neighbors

In [10]:
from sklearn.neighbors import KNeighborsClassifier

In [11]:
knn = KNeighborsClassifier()
knn.fit(X_train,y_train)
print(knn.score(X_train,y_train),knn.score(X_val,y_val))

0.9771726190476191 0.9648809523809524


### 2. Support Vector Machine

In [12]:
# from sklearn.svm import SVC
# sv = SVC()
# sv.fit(X_train,y_train)
# print(sv.score(X_train,y_train),sv.score(X_val,y_val))

### 3. RandomForest Classifier

In [13]:
from sklearn.ensemble import RandomForestClassifier
rf =RandomForestClassifier()
rf.fit(X_train,y_train)
print(rf.score(X_train,y_train),rf.score(X_val,y_val))

1.0 0.9644047619047619


### XgBoost Classifier

In [14]:
# import xgboost as xgb
# model = xgb.XGBClassifier()
# model.fit(X_train,y_train)
# print(model.score(X_train,y_train),model.score(X_val,y_val))

## Training the CNN Model

In [15]:
import tensorflow as tf
import pandas as pd
import numpy as np

In [16]:
train_df = pd.read_csv('/kaggle/input/digit-recognizer/train.csv')
test_df = pd.read_csv('/kaggle/input/digit-recognizer/test.csv')

In [17]:
X = train_df.drop(['label'],axis = 1).astype('int32')
y = train_df['label'].astype('float32')
x_test = test_df.astype('float32')
X.shape, y.shape, x_test.shape

((42000, 784), (42000,), (28000, 784))

In [18]:
X = X.values.reshape(-1, 28, 28, 1) / 255.0
x_test = x_test.values.reshape(-1, 28, 28, 1) / 255.0
X.shape, x_test.shape

((42000, 28, 28, 1), (28000, 28, 28, 1))

In [19]:
y = tf.keras.utils.to_categorical(y, num_classes=10)

In [20]:
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

### Building the Neural Network

In [21]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Flatten, Dropout, BatchNormalization

In [22]:
model = Sequential()
model.add(Conv2D(16, (3,3), 1, activation='relu', input_shape=(28,28,1)))
model.add(MaxPooling2D())

model.add(Conv2D(32, (3,3), 1, activation='relu'))
model.add(MaxPooling2D())
model.add(Conv2D(16, (3,3), 1, activation='relu'))
model.add(MaxPooling2D())

model.add(Flatten())
model.add(Dense(256, activation='relu'))
model.add(Dense(10, activation='softmax'))

In [23]:
model.compile('adam', loss=tf.losses.CategoricalCrossentropy(), metrics=['accuracy'])

In [24]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 26, 26, 16)        160       
                                                                 
 max_pooling2d (MaxPooling2D  (None, 13, 13, 16)       0         
 )                                                               
                                                                 
 conv2d_1 (Conv2D)           (None, 11, 11, 32)        4640      
                                                                 
 max_pooling2d_1 (MaxPooling  (None, 5, 5, 32)         0         
 2D)                                                             
                                                                 
 conv2d_2 (Conv2D)           (None, 3, 3, 16)          4624      
                                                                 
 max_pooling2d_2 (MaxPooling  (None, 1, 1, 16)         0

### Data Augmentation and Early Stopping

In [25]:
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [26]:
# Define data augmentation parameters
datagen = ImageDataGenerator(
    rotation_range=10,  # rotate images randomly within 10 degrees
    zoom_range=0.1,     # zoom images randomly within 10%
    width_shift_range=0.1,  # shift images horizontally randomly within 10%
    height_shift_range=0.1, # shift images vertically randomly within 10%
    shear_range=0.1,    # shear images randomly within 10 degrees
    horizontal_flip=False, # flip images horizontally randomly
    vertical_flip=False   # flip images vertically randomly
)

# Fit the augmentation method on training set
datagen.fit(X_train)

In [27]:
# Define early stopping callback
early_stop = EarlyStopping(monitor='val_loss', patience=5)

# Train the model using data augmentation
history = model.fit(
    datagen.flow(X_train, y_train, batch_size=64),
    steps_per_epoch=len(X_train)/64, 
    epochs=50, 
    validation_data=(X_val, y_val),
#     callbacks=[early_stop],
    verbose=1
)

Epoch 1/50
525/525 [==============================] - 23s 25ms/step - loss: 0.8365 - accuracy: 0.7259 - val_loss: 0.3291 - val_accuracy: 0.8989
Epoch 2/50
525/525 [==============================] - 13s 25ms/step - loss: 0.3881 - accuracy: 0.8774 - val_loss: 0.2113 - val_accuracy: 0.9358
Epoch 3/50
525/525 [==============================] - 14s 26ms/step - loss: 0.2968 - accuracy: 0.9071 - val_loss: 0.1821 - val_accuracy: 0.9445
Epoch 4/50
525/525 [==============================] - 13s 26ms/step - loss: 0.2531 - accuracy: 0.9207 - val_loss: 0.1609 - val_accuracy: 0.9496
Epoch 5/50
525/525 [==============================] - 14s 26ms/step - loss: 0.2206 - accuracy: 0.9311 - val_loss: 0.1251 - val_accuracy: 0.9620
Epoch 6/50
525/525 [==============================] - 14s 26ms/step - loss: 0.2053 - accuracy: 0.9346 - val_loss: 0.1076 - val_accuracy: 0.9654
Epoch 7/50
525/525 [==============================] - 13s 25ms/step - loss: 0.1867 - accuracy: 0.9406 - val_loss: 0.1189 - val_accuracy:

In [28]:
# res = model.predict(x_test)
# res = np.argmax(res,axis = 1)
# results_2 = pd.Series(res[0],name="Label")
# submission_df = pd.DataFrame(results_2)
# submission_df["ImageId"] = range(1,28001)
# submission_df = submission_df[["ImageId", "Label"]]
# submission_df

## MNIST CNN NETs Ensemble

In [29]:
# BUILD CONVOLUTIONAL NEURAL NETWORKS
nets = 3
model = [0] *nets
for j in range(nets):
    model[j] = Sequential()
    model[j].add(Conv2D(32, kernel_size = 3, activation='relu', input_shape = (28, 28, 1)))
    model[j].add(BatchNormalization())
    model[j].add(Conv2D(32, kernel_size = 3, activation='relu'))
    model[j].add(BatchNormalization())
    model[j].add(Conv2D(32, kernel_size = 5, strides=2, padding='same', activation='relu'))
    model[j].add(BatchNormalization())
    model[j].add(Dropout(0.4))
    model[j].add(Conv2D(64, kernel_size = 3, activation='relu'))
    model[j].add(BatchNormalization())
    model[j].add(Conv2D(64, kernel_size = 3, activation='relu'))
    model[j].add(BatchNormalization())
    model[j].add(Conv2D(64, kernel_size = 5, strides=2, padding='same', activation='relu'))
    model[j].add(BatchNormalization())
    model[j].add(Dropout(0.4))
    model[j].add(Conv2D(128, kernel_size = 4, activation='relu'))
    model[j].add(BatchNormalization())
    model[j].add(Flatten())
    model[j].add(Dropout(0.4))
    model[j].add(Dense(10, activation='softmax'))
    # COMPILE WITH ADAM OPTIMIZER AND CROSS ENTROPY COST
    model[j].compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

In [30]:
from keras.callbacks import LearningRateScheduler
# DECREASE LEARNING RATE EACH EPOCH
annealer = LearningRateScheduler(lambda x: 1e-3 * 0.95 ** x)
# TRAIN NETWORKS
history = [0] * nets
epochs = 30
for j in range(nets):
    X_train2, X_val2, Y_train2, Y_val2 = train_test_split(X_train, y_train, test_size = 0.1)
    history[j] = model[j].fit_generator(datagen.flow(X_train2,Y_train2, batch_size=64),
        epochs = epochs, steps_per_epoch = X_train2.shape[0]//64,  
        validation_data = (X_val2,Y_val2), callbacks=[annealer], verbose=1)
    print("CNN {0:d}: Epochs={1:d}, Train accuracy={2:.5f}, Validation accuracy={3:.5f}".format(
        j+1,epochs,max(history[j].history['accuracy']),max(history[j].history['val_accuracy']) ))

Epoch 1/30


/opt/conda/lib/python3.7/site-packages/ipykernel_launcher.py:11: UserWarning: `Model.fit_generator` is deprecated and will be removed in a future version. Please use `Model.fit`, which supports generators.
  # This is added back by InteractiveShellApp.init_path()
2023-03-19 07:00:18.820838: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:954] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape insequential_1/dropout/dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


472/472 [==============================] - 19s 29ms/step - loss: 0.5398 - accuracy: 0.8325 - val_loss: 4.4886 - val_accuracy: 0.1771 - lr: 0.0010
Epoch 2/30
472/472 [==============================] - 14s 29ms/step - loss: 0.1542 - accuracy: 0.9533 - val_loss: 0.0425 - val_accuracy: 0.9860 - lr: 9.5000e-04
Epoch 3/30
472/472 [==============================] - 13s 28ms/step - loss: 0.1169 - accuracy: 0.9639 - val_loss: 0.0438 - val_accuracy: 0.9869 - lr: 9.0250e-04
Epoch 4/30
472/472 [==============================] - 14s 29ms/step - loss: 0.1017 - accuracy: 0.9697 - val_loss: 0.0502 - val_accuracy: 0.9857 - lr: 8.5737e-04
Epoch 5/30
472/472 [==============================] - 13s 28ms/step - loss: 0.0879 - accuracy: 0.9728 - val_loss: 0.0417 - val_accuracy: 0.9857 - lr: 8.1451e-04
Epoch 6/30
472/472 [==============================] - 14s 29ms/step - loss: 0.0778 - accuracy: 0.9773 - val_loss: 0.0325 - val_accuracy: 0.9878 - lr: 7.7378e-04
Epoch 7/30
472/472 [=============================

2023-03-19 07:09:09.913047: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:954] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape insequential_2/dropout_3/dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


472/472 [==============================] - 18s 29ms/step - loss: 0.5500 - accuracy: 0.8323 - val_loss: 2.0188 - val_accuracy: 0.4771 - lr: 0.0010
Epoch 2/30
472/472 [==============================] - 14s 29ms/step - loss: 0.1679 - accuracy: 0.9488 - val_loss: 0.0576 - val_accuracy: 0.9842 - lr: 9.5000e-04
Epoch 3/30
472/472 [==============================] - 14s 30ms/step - loss: 0.1131 - accuracy: 0.9650 - val_loss: 0.0481 - val_accuracy: 0.9854 - lr: 9.0250e-04
Epoch 4/30
472/472 [==============================] - 13s 28ms/step - loss: 0.0985 - accuracy: 0.9700 - val_loss: 0.0620 - val_accuracy: 0.9798 - lr: 8.5737e-04
Epoch 5/30
472/472 [==============================] - 14s 29ms/step - loss: 0.0873 - accuracy: 0.9727 - val_loss: 0.0392 - val_accuracy: 0.9893 - lr: 8.1451e-04
Epoch 6/30
472/472 [==============================] - 13s 28ms/step - loss: 0.0749 - accuracy: 0.9783 - val_loss: 0.0260 - val_accuracy: 0.9920 - lr: 7.7378e-04
Epoch 7/30
472/472 [=============================

2023-03-19 07:17:28.969496: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:954] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape insequential_3/dropout_6/dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


472/472 [==============================] - 18s 29ms/step - loss: 0.5428 - accuracy: 0.8332 - val_loss: 0.1671 - val_accuracy: 0.9455 - lr: 0.0010
Epoch 2/30
472/472 [==============================] - 14s 29ms/step - loss: 0.1532 - accuracy: 0.9538 - val_loss: 0.0547 - val_accuracy: 0.9818 - lr: 9.5000e-04
Epoch 3/30
472/472 [==============================] - 14s 29ms/step - loss: 0.1153 - accuracy: 0.9649 - val_loss: 0.0395 - val_accuracy: 0.9851 - lr: 9.0250e-04
Epoch 4/30
472/472 [==============================] - 14s 30ms/step - loss: 0.0992 - accuracy: 0.9705 - val_loss: 0.0420 - val_accuracy: 0.9866 - lr: 8.5737e-04
Epoch 5/30
472/472 [==============================] - 14s 29ms/step - loss: 0.0781 - accuracy: 0.9769 - val_loss: 0.0422 - val_accuracy: 0.9863 - lr: 8.1451e-04
Epoch 6/30
472/472 [==============================] - 14s 29ms/step - loss: 0.0719 - accuracy: 0.9775 - val_loss: 0.0483 - val_accuracy: 0.9842 - lr: 7.7378e-04
Epoch 7/30
472/472 [=============================

In [31]:
# ENSEMBLE PREDICTIONS AND SUBMIT
results = np.zeros( (x_test.shape[0],10) ) 
for j in range(nets):
    resultsi = model[j].predict(x_test)
    results = results + resultsi
    
    resultsi = np.argmax(resultsi,axis = 1)
    resultsi = pd.Series(resultsi,name="Label")
    submissioni = pd.concat([pd.Series(range(1,28001),name = "ImageId"),resultsi],axis = 1)
    submissioni.to_csv("SUBMISSION"+str(j+1)+".csv",index=False)
    
results = np.argmax(results,axis = 1)
results = pd.Series(results,name="Label")
submission = pd.concat([pd.Series(range(1,28001),name = "ImageId"),results],axis = 1)
submission.to_csv("ENSEMBLE.csv",index=False)

875/875 [==============================] - 3s 3ms/step
